# Demo F: Inference Engines Comparison

**Platform:** Lightning.ai Studio (A100 GPU)

**Goal:** Run the SAME test across all engines. See how each one stacks optimizations.

## What each engine brings:
| Engine | Key Innovations | Best For |
|--------|----------------|----------|
| HuggingFace | None (baseline) | Prototyping |
| vLLM | PagedAttention + continuous batching | General production |
| SGLang | RadixAttention (tree prefix cache) + constrained decoding | Agents, multi-turn |
| TensorRT-LLM | AOT compile + CUDA graphs + XQA kernels + FP8 | Max throughput (NVIDIA only) |

## How this demo works:
1. Run HF baseline
2. Start vLLM, run same test
3. Start vLLM with prefix caching, show TTFT improvement
4. Start SGLang, show RadixAttention advantage
5. Compare all results in one chart

## Setup

In [ ]:
# Cell 1: Install everything
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'torch', 'transformers', 'accelerate', 'matplotlib',
                       'requests', 'tqdm', 'numpy<2', 'scipy>=1.14'])

import torch, time, requests
import matplotlib.pyplot as plt
from tqdm import tqdm

# ─── SHARED TEST CONFIG ────────────────────────────────
MODEL = 'mistralai/Mistral-7B-v0.1'
PORT = 8000
BASE_URL = f'http://localhost:{PORT}/v1'

N_REQUESTS = 5
N_TOKENS = 10
PROMPTS = [f'Explain machine learning concept {i} in simple terms:' for i in range(N_REQUESTS)]

# For prefix caching test: shared system prompt
SYSTEM_PROMPT = 'You are a helpful AI assistant. Answer concisely. ' * 20  # ~100 tokens
PREFIX_PROMPTS = [f'{SYSTEM_PROMPT} Question {i}: What is {i}+{i}?' for i in range(N_REQUESTS)]

# Store results for final comparison
results = {}  # {'engine_name': {'throughput': X, 'ttft': Y}}
# ──────────────────────────────────────────────────────

def send_requests(prompts, max_tokens, label):
    """Send prompts to running server, return (elapsed_s, throughput_tok_s)."""
    # Warmup
    requests.post(f'{BASE_URL}/completions', json={'model': MODEL, 'prompt': 'warmup', 'max_tokens': 1, 'temperature': 0})
    # Benchmark
    t0 = time.perf_counter()
    for p in tqdm(prompts, desc=label):
        requests.post(f'{BASE_URL}/completions', json={'model': MODEL, 'prompt': p, 'max_tokens': max_tokens, 'temperature': 0})
    elapsed = time.perf_counter() - t0
    throughput = (len(prompts) * max_tokens) / elapsed
    print(f'  {label}: {elapsed:.2f}s, {throughput:.0f} tok/s')
    return elapsed, throughput

print(f'Config: {N_REQUESTS} requests x {N_TOKENS} tokens')
print(f'Prefix test: {len(PREFIX_PROMPTS)} requests with shared ~100 token prefix')

## Part 1: HuggingFace Baseline

No engine. Sequential generation. This is the floor.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print('Loading Mistral-7B...')
hf_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='auto', token=False)
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL, token=False)

# Warmup (3 calls)
print('Warming up...')
for wi in range(3):
    with torch.no_grad():
        w = hf_tokenizer(f'Warmup {wi}', return_tensors='pt').to('cuda')
        hf_model.generate(**w, max_new_tokens=5, pad_token_id=hf_tokenizer.eos_token_id)
torch.cuda.synchronize()

# Benchmark
print(f'Running {N_REQUESTS} requests x {N_TOKENS} tokens...')
torch.cuda.synchronize()
hf_start = time.perf_counter()
for p in tqdm(PROMPTS, desc='HF'):
    inp = hf_tokenizer(p, return_tensors='pt').to('cuda')
    with torch.no_grad():
        hf_model.generate(**inp, max_new_tokens=N_TOKENS, do_sample=False, pad_token_id=hf_tokenizer.eos_token_id)
torch.cuda.synchronize()
hf_time = time.perf_counter() - hf_start
hf_throughput = (N_REQUESTS * N_TOKENS) / hf_time

results['HuggingFace'] = {'throughput': hf_throughput, 'time': hf_time}
print(f'\nHF: {hf_time:.1f}s, {hf_throughput:.0f} tok/s')

del hf_model, hf_tokenizer
torch.cuda.empty_cache()
print('GPU freed.')

## Part 2: vLLM (PagedAttention + Continuous Batching)

Start in terminal:
```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --gpu-memory-utilization 0.90 \
    --port 8000
```

In [ ]:
# Verify server
try:
    r = requests.get(f'{BASE_URL}/models')
    print(f'vLLM ready: {r.json()["data"][0]["id"]}')
except:
    print('Start vLLM server first!')

In [ ]:
# Same test on vLLM
vllm_elapsed, vllm_throughput = send_requests(PROMPTS, N_TOKENS, 'vLLM')
results['vLLM'] = {'throughput': vllm_throughput, 'time': vllm_elapsed}
print(f'\nSpeedup over HF: {vllm_throughput/results["HuggingFace"]["throughput"]:.1f}x')

## Part 3: vLLM + Prefix Caching

Restart with:
```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --enable-prefix-caching \
    --gpu-memory-utilization 0.90 \
    --port 8000
```

In [ ]:
# Prefix caching test: shared system prompt across requests
# First batch: cold (prefix not cached)
print('Cold batch (prefix not yet cached)...')
cold_elapsed, cold_tp = send_requests(PREFIX_PROMPTS[:3], 1, 'Cold')

# Second batch: warm (prefix now cached)
print('Warm batch (prefix cached from first batch)...')
warm_elapsed, warm_tp = send_requests(PREFIX_PROMPTS[3:], 1, 'Warm')

cold_ttft = (cold_elapsed / 3) * 1000  # ms per request
warm_ttft = (warm_elapsed / 2) * 1000

results['vLLM+Prefix'] = {'throughput': vllm_throughput, 'cold_ttft': cold_ttft, 'warm_ttft': warm_ttft}
print(f'\nCold TTFT: {cold_ttft:.0f} ms')
print(f'Warm TTFT: {warm_ttft:.0f} ms')
print(f'Prefix cache speedup: {cold_ttft/warm_ttft:.1f}x faster TTFT')

## Part 3b: vLLM + KV Quantization

Restart with FP8 KV cache (stores K,V in 8 bits instead of 16):
```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --enable-prefix-caching \
    --kv-cache-dtype fp8 \
    --gpu-memory-utilization 0.90 \
    --port 8000
```

Benefit: 2x more users fit in memory (half the KV cache size).

In [ ]:
# KV Quantization: stress test with more concurrent requests
# With FP8 KV, the same GPU holds 2x more users
STRESS_PROMPTS = [f'Explain topic {i} in detail:' for i in range(20)]  # 20 users

print('Stress test: 20 concurrent requests (tests memory capacity)...')
kvq_elapsed, kvq_throughput = send_requests(STRESS_PROMPTS, N_TOKENS, 'vLLM+FP8 KV (20 users)')

results['vLLM+KV Quant'] = {'throughput': kvq_throughput, 'time': kvq_elapsed}
print(f'\n20 users served without OOM. FP8 KV = 2x memory for KV cache.')


## Part 3c: vLLM + Speculative Decoding

Restart with draft model:
```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --speculative-model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --num-speculative-tokens 5 \
    --gpu-memory-utilization 0.90 \
    --port 8000
```

Needs ~20GB (both models loaded). Draft generates 5 tokens, main verifies in 1 pass.

**Skip if OOM.** A10G (24GB) is tight with 2 models.

In [ ]:
# Speculative decoding: test with longer output to see decode speedup
SPEC_PROMPTS = [f'Write a paragraph about topic {i}:' for i in range(3)]  # fewer requests, more tokens
SPEC_TOKENS = 50  # longer output to see decode benefit

print(f'Speculative decode: 3 requests x 50 tokens...')
try:
    spec_elapsed, spec_throughput = send_requests(SPEC_PROMPTS, SPEC_TOKENS, 'vLLM+Speculative')
    results['vLLM+Speculative'] = {'throughput': spec_throughput, 'time': spec_elapsed}
    print(f'\nDraft model generates 5 candidates, main verifies in 1 pass.')
    print(f'Effective decode speed: {spec_throughput:.0f} tok/s')
except Exception as e:
    print(f'Speculative decoding failed (likely OOM with 2 models): {e}')
    print('Skip this on A10G. Works on A100-80.')


## Part 4: SGLang (RadixAttention)

Kill vLLM, then start SGLang:
```bash
python -m sglang.launch_server \
    --model-path mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --port 8000
```

SGLang has prefix caching ON by default via RadixAttention (tree, not hash).

**Note:** If SGLang install fails, skip this section. The concept is explained in slides.

In [ ]:
# SGLang test (same API format)
try:
    r = requests.get(f'{BASE_URL}/models')
    print(f'SGLang ready: {r.json()}')
    
    # Throughput test
    sgl_elapsed, sgl_throughput = send_requests(PROMPTS, N_TOKENS, 'SGLang')
    
    # Prefix test (automatic in SGLang)
    print('\nPrefix test (RadixAttention, always on)...')
    sgl_cold_e, _ = send_requests(PREFIX_PROMPTS[:3], 1, 'SGLang Cold')
    sgl_warm_e, _ = send_requests(PREFIX_PROMPTS[3:], 1, 'SGLang Warm')
    sgl_cold_ttft = (sgl_cold_e / 3) * 1000
    sgl_warm_ttft = (sgl_warm_e / 2) * 1000
    
    results['SGLang'] = {'throughput': sgl_throughput, 'cold_ttft': sgl_cold_ttft, 'warm_ttft': sgl_warm_ttft}
    print(f'SGLang prefix speedup: {sgl_cold_ttft/sgl_warm_ttft:.1f}x')
except:
    print('SGLang not available. Skipping.')

## Final Comparison

All engines on the same test. Same model. Same GPU.

In [ ]:
# Build comparison chart from whatever results we have
engines = list(results.keys())
throughputs = [results[e]['throughput'] for e in engines]

fig, ax = plt.subplots(figsize=(9, 5))
colors = {'HuggingFace': '#ffe4e6', 'vLLM': '#dcfce7', 'vLLM+Prefix': '#dbeafe', 'SGLang': '#f3e8ff'}
bar_colors = [colors.get(e, '#f3f4f6') for e in engines]

bars = ax.bar(engines, throughputs, color=bar_colors, edgecolor='#000', linewidth=1.2)

# Labels + speedup
baseline = results['HuggingFace']['throughput']
for bi, (eng, tp) in enumerate(zip(engines, throughputs)):
    ax.text(bi, tp + max(throughputs)*0.02, f'{tp:.0f} tok/s', ha='center', fontsize=10)
    if eng != 'HuggingFace':
        ax.text(bi, tp*0.5, f'{tp/baseline:.1f}x', ha='center', fontsize=13, color='#166534', fontweight='bold')

ax.set_ylabel('Throughput (tok/s)', fontsize=11)
ax.set_title(f'Engine Comparison: {N_REQUESTS} requests x {N_TOKENS} tokens, same A100',
             fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# Prefix caching comparison if available
prefix_engines = [e for e in results if 'cold_ttft' in results[e]]
if prefix_engines:
    print(f'\n--- Prefix Caching TTFT ---')
    for e in prefix_engines:
        r = results[e]
        print(f'  {e}: Cold {r["cold_ttft"]:.0f}ms -> Warm {r["warm_ttft"]:.0f}ms ({r["cold_ttft"]/r["warm_ttft"]:.1f}x)')

## When to Use What

| Use Case | Engine | Why |
|----------|--------|-----|
| General production | vLLM | PagedAttention + batching, broadest hardware support |
| Agents with system prompts | SGLang | RadixAttention tree, 5x better prefix reuse |
| Max throughput (NVIDIA) | TensorRT-LLM | AOT compile, CUDA graphs, FP8, XQA kernels |
| Prototyping/debugging | HuggingFace | Simple, no server needed |

**Note on TensorRT-LLM:** Requires offline compilation step. Cannot benchmark live in 2 minutes.
See Lightning.ai template: https://lightning.ai/docs/examples/deploy-full-code/tensorrt-llm